Attention variants

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):

    def __init__(self,d_model,n_heads,d_head):

        super().__init__()
        self.d_model=d_model
        self.n_heads=n_heads
        self.d_head= d_model//n_heads

        self.qkv_proj = nn.Linear(d_model,3*d_model)

        self.out_proj= nn.Linear(d_model,d_model)

        self.scale = 1.0 /math.sqrt(d_head)


    def forward(self,x,mask):

        B,S,D = x.shape # [B,S,D]

        qkv=self.qkv_proj(x) # [B,S,3D]

        q,k,v= qkv.chunk(3,dim=-1) # [B,S,D]


        q_t = q.view(B,S,self.n_heads,self.d_head) # [B,S,H,D_H]
        k_t = k.view(B,S,self.n_heads,self.d_head) # [B,S,H,D_H]
        v_t = v.view(B,S,self.n_heads,self.d_head) # [B,S,H,D_H]

        q_t=q_t.permute(0,2,1,3) #[B,H,S,D_H]
        k_t=k_t.permute(0,2,1,3) #[B,H,S,D_H]
        v_t=v_t.permute(0,2,1,3) #[B,H,S,D_H]

        attn_scores = torch.matmul(q_t, torch.transpose(k_t, 2,3))*self.scale # [B,H,S,S]


        if mask:

            attn_scores = attn_scores.masked_fill(mask==0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1) # [B,H,S,S]

        out = torch.matmul(attn_weights,v) # [B,H,S,S]*[B,H,S,D_H]
         # out: [B, H, S, D_h]

        # premute again 
        out = out.permute(0,2,1,3) # [B,S,H,D_H]
        out = out.contiguous().view(B,S,self.d_model) # [B,S,D]

        res = self.out_proj(out) 

